# Day 13 — Compare Deep Learning Models Against ML Baselines

**Project:** IPL Player Performance Analytics and Prediction Using Machine Learning and Deep Learning

## Objectives
- Load Day 8 tree-based baseline metrics, Day 10 selected classical model test metrics, and Day 12 LSTM/GRU results.
- Compare the three prediction targets using MAE, RMSE and R².
- Check test-period compatibility before interpreting the comparison.
- Generate comparison reports and documentation from the actual files.

**Important:** no results are hardcoded. If files or schemas differ from expectations, the notebook flags that limitation rather than inventing scores.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

current_dir = Path.cwd().resolve()
candidates = [current_dir, *current_dir.parents]
def is_project_root(p):
    return (p / "reports" / "day8").is_dir() or (
        (p / "reports").is_dir() and (p / "data" / "processed").is_dir()
    )
PROJECT_ROOT = next((p for p in candidates if is_project_root(p)), None)
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path(r"D:\Git\Personal\ipl-player-performance")

REPORT_DIR = PROJECT_ROOT / "reports"
DAY8_DIR, DAY10_DIR, DAY12_DIR = REPORT_DIR / "day8", REPORT_DIR / "day10", REPORT_DIR / "day12"
DAY13_DIR, DOCS_DIR = REPORT_DIR / "day13", PROJECT_ROOT / "docs"
for folder in [DAY13_DIR, DOCS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

PATHS = {
    "day8_metrics": DAY8_DIR / "model_comparison_metrics.csv",
    "day8_selection": DAY8_DIR / "validation_model_selection.csv",
    "day10_metrics": DAY10_DIR / "final_candidate_test_metrics.csv",
    "day10_splits": DAY10_DIR / "final_split_summary.csv",
    "day12_metrics": DAY12_DIR / "selected_lstm_gru_test_metrics.csv",
    "day12_selection": DAY12_DIR / "validation_architecture_comparison.csv",
    "day12_splits": DAY12_DIR / "chronological_split_summary.csv",
}
print("Project root:", PROJECT_ROOT)
for name, path in PATHS.items():
    print(("FOUND  " if path.is_file() else "MISSING"), name, "->", path)

Project root: D:\Git\Personal\ipl-player-performance
FOUND   day8_metrics -> D:\Git\Personal\ipl-player-performance\reports\day8\model_comparison_metrics.csv
FOUND   day8_selection -> D:\Git\Personal\ipl-player-performance\reports\day8\validation_model_selection.csv
FOUND   day10_metrics -> D:\Git\Personal\ipl-player-performance\reports\day10\final_candidate_test_metrics.csv
FOUND   day10_splits -> D:\Git\Personal\ipl-player-performance\reports\day10\final_split_summary.csv
FOUND   day12_metrics -> D:\Git\Personal\ipl-player-performance\reports\day12\selected_lstm_gru_test_metrics.csv
FOUND   day12_selection -> D:\Git\Personal\ipl-player-performance\reports\day12\validation_architecture_comparison.csv
FOUND   day12_splits -> D:\Git\Personal\ipl-player-performance\reports\day12\chronological_split_summary.csv


## 1. Load reports and inspect their schemas

The Day 10 and Day 12 reports are the primary final-test comparison sources. Day 8 results provide context about the original baseline models and validation selection.

In [2]:
def read_optional(path):
    return pd.read_csv(path) if path.is_file() else None

frames = {name: read_optional(path) for name, path in PATHS.items()}
source_status = pd.DataFrame([
    {"source": name, "path": str(PATHS[name]),
     "status": "FOUND" if frame is not None else "MISSING",
     "columns": ", ".join(map(str, frame.columns)) if frame is not None else ""}
    for name, frame in frames.items()
])
display(source_status)
for name, frame in frames.items():
    if frame is not None:
        print(f"\n{name}: {len(frame)} rows")
        display(frame.head(3))

,source,path,status,columns
0,day8_metrics,D:\Git\Personal\ipl-player-performance\reports...,FOUND,"dataset, target, model, split, feature_count, ..."
1,day8_selection,D:\Git\Personal\ipl-player-performance\reports...,FOUND,"dataset, target, selected_by_validation_mae, v..."
2,day10_metrics,D:\Git\Personal\ipl-player-performance\reports...,FOUND,"dataset, target, selected_model_from_day8, fea..."
3,day10_splits,D:\Git\Personal\ipl-player-performance\reports...,FOUND,"dataset, target, all_usable_rows, train_rows, ..."
4,day12_metrics,D:\Git\Personal\ipl-player-performance\reports...,FOUND,"dataset, target, selected_architecture, train_..."
5,day12_selection,D:\Git\Personal\ipl-player-performance\reports...,FOUND,"dataset, target, architecture, epochs_run, val..."
6,day12_splits,D:\Git\Personal\ipl-player-performance\reports...,FOUND,"dataset, target, split, rows, target_date_star..."



day8_metrics: 24 rows


,dataset,target,model,split,feature_count,train_rows,validation_rows,test_rows,MAE,RMSE,R2
0,batting,next_match_runs,DummyMedian,validation,15,12621,2704,2705,16.475222,23.850519,-0.129310
1,batting,next_match_runs,DummyMedian,test,15,12621,2704,2705,17.912754,26.069777,-0.188754
2,batting,next_match_runs,LinearRegression,validation,15,12621,2704,2705,15.864845,21.231884,0.105058



day8_selection: 3 rows


,dataset,target,selected_by_validation_mae,validation_mae,saved_pipeline
0,batting,next_match_runs,LinearRegression,15.864845,D:\Git\Personal\ipl-player-performance\models\...
1,bowling,next_match_wickets,LinearRegression,0.789086,D:\Git\Personal\ipl-player-performance\models\...
2,bowling,next_match_runs_conceded,XGBoost,8.490233,D:\Git\Personal\ipl-player-performance\models\...


...

,dataset,target,selected_architecture,train_samples,validation_samples,test_samples,feature_count,test_MAE,test_RMSE,test_R2,test_mean_residual_actual_minus_predicted
0,batting,next_match_runs,LSTM,11074,2373,2374,15,16.700930,22.505660,0.105188,2.578387
1,bowling,next_match_wickets,LSTM,8634,1850,1851,15,0.782648,0.999579,0.019520,0.029028
2,bowling,next_match_runs_conceded,GRU,8634,1850,1851,15,9.509530,11.931938,0.049363,3.587987



day12_selection: 6 rows


,dataset,target,architecture,epochs_run,validation_MAE,validation_RMSE,validation_R2
0,batting,next_match_runs,LSTM,36,16.311644,21.884249,0.106725
1,batting,next_match_runs,GRU,48,16.338200,21.845088,0.109919
2,bowling,next_match_wickets,LSTM,17,0.802349,1.040258,0.026574



day12_splits: 9 rows


,dataset,target,split,rows,target_date_start,target_date_end
0,batting,next_match_runs,train,11074,2008-05-03,2022-04-10
1,batting,next_match_runs,validation,2373,2022-04-10,2024-04-29
2,batting,next_match_runs,test,2374,2024-04-29,2026-05-31


## 2. Normalize available metric tables

Column names may vary between notebook versions. This cell recognizes common variants and reports a source as unavailable if target, MAE, or RMSE cannot be found.

In [3]:
def pick_col(df, options):
    return next((c for c in options if c in df.columns), None)

def normalize_metrics(df, source, model_options, mae_options, rmse_options, r2_options):
    if df is None:
        return pd.DataFrame()
    target_col = pick_col(df, ["target"])
    model_col = pick_col(df, model_options)
    mae_col = pick_col(df, mae_options)
    rmse_col = pick_col(df, rmse_options)
    r2_col = pick_col(df, r2_options)
    if target_col is None or mae_col is None or rmse_col is None:
        print(f"SCHEMA REVIEW: {source}; columns found: {list(df.columns)}")
        return pd.DataFrame()
    return pd.DataFrame({
        "target": df[target_col].astype(str),
        "model": df[model_col].astype(str) if model_col else source,
        "MAE": pd.to_numeric(df[mae_col], errors="coerce"),
        "RMSE": pd.to_numeric(df[rmse_col], errors="coerce"),
        "R2": pd.to_numeric(df[r2_col], errors="coerce") if r2_col else np.nan,
        "source": source
    })

day8_metrics = normalize_metrics(
    frames["day8_metrics"], "Day 8 metrics",
    ["model", "model_name", "model_type", "estimator"],
    ["MAE", "mae", "test_MAE", "test_mae"],
    ["RMSE", "rmse", "test_RMSE", "test_rmse"],
    ["R2", "r2", "test_R2", "test_r2"]
)
day10_metrics = normalize_metrics(
    frames["day10_metrics"], "Day 10 final candidate",
    ["selected_model_from_day8", "selected_model", "model"],
    ["MAE", "test_MAE", "test_mae"],
    ["RMSE", "test_RMSE", "test_rmse"],
    ["R2", "test_R2", "test_r2"]
)
day12_metrics = normalize_metrics(
    frames["day12_metrics"], "Day 12 final candidate",
    ["selected_architecture", "architecture", "model"],
    ["test_MAE", "MAE", "test_mae"],
    ["test_RMSE", "RMSE", "test_rmse"],
    ["test_R2", "R2", "test_r2"]
)
print("Normalized rows — Day 8:", len(day8_metrics), "Day 10:", len(day10_metrics), "Day 12:", len(day12_metrics))

Normalized rows — Day 8: 24 Day 10: 3 Day 12: 3


## 3. Compare final candidate models by target

For each target, the Day 10 selected classical model and Day 12 selected deep-learning model are shown side by side. The Day 8 table is retained as baseline context, but validation metrics must not be compared directly against test metrics.

In [4]:
targets = ["next_match_runs", "next_match_wickets", "next_match_runs_conceded"]
comparison_rows = []
for target in targets:
    for family, df in [("Classical ML", day10_metrics), ("Deep learning", day12_metrics)]:
        match = df[df["target"] == target] if not df.empty else pd.DataFrame()
        if match.empty:
            comparison_rows.append({
                "target": target, "model_family": family, "model": "NOT AVAILABLE",
                "MAE": np.nan, "RMSE": np.nan, "R2": np.nan,
                "source": "Missing or incompatible report"
            })
        else:
            row = match.iloc[0]
            comparison_rows.append({
                "target": target, "model_family": family, "model": row["model"],
                "MAE": row["MAE"], "RMSE": row["RMSE"], "R2": row["R2"],
                "source": row["source"]
            })
comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)

print("\nDay 8 baseline metrics (context; verify metric split before comparing):")
display(day8_metrics)

,target,model_family,model,MAE,RMSE,R2,source
0,next_match_runs,Classical ML,LinearRegression,16.850733,22.619524,0.105080,Day 10 final candidate
1,next_match_runs,Deep learning,LSTM,16.700930,22.505660,0.105188,Day 12 final candidate
2,next_match_wickets,Classical ML,LinearRegression,0.767798,0.989258,0.026481,Day 10 final candidate
3,next_match_wickets,Deep learning,LSTM,0.782648,0.999579,0.019520,Day 12 final candidate
4,next_match_runs_conceded,Classical ML,XGBoost,9.516070,12.015263,0.053118,Day 10 final candidate
5,next_match_runs_conceded,Deep learning,GRU,9.509530,11.931938,0.049363,Day 12 final candidate



Day 8 baseline metrics (context; verify metric split before comparing):


,target,model,MAE,RMSE,R2,source
0,next_match_runs,DummyMedian,16.475222,23.850519,-0.129310,Day 8 metrics
1,next_match_runs,DummyMedian,17.912754,26.069777,-0.188754,Day 8 metrics
2,next_match_runs,LinearRegression,15.864845,21.231884,0.105058,Day 8 metrics
3,next_match_runs,LinearRegression,16.835979,22.663426,0.101603,Day 8 metrics
4,next_match_runs,RandomForest,16.176661,21.458592,0.085844,Day 8 metrics
5,next_match_runs,RandomForest,17.008145,22.884593,0.083983,Day 8 metrics
6,next_match_runs,XGBoost,15.931191,21.312452,0.098254,Day 8 metrics
7,next_match_runs,XGBoost,16.918188,22.788992,0.091620,Day 8 metrics
8,next_match_wickets,DummyMedian,0.789424,1.051551,-0.000545,Day 8 metrics
9,next_match_wickets,DummyMedian,0.759320,1.002828,-0.000410,Day 8 metrics


## 4. Check whether the test periods are compatible

Matching date ranges are necessary but not sufficient: sample counts and player/match populations should also be checked. The notebook will avoid declaring a winner when the reports do not establish comparability.

In [5]:
def split_range(df, label):
    if df is None or df.empty:
        return {"source": label, "status": "MISSING", "test_start": None, "test_end": None}
    start_col = pick_col(df, ["test_start", "test_start_date", "target_date_start"])
    end_col = pick_col(df, ["test_end", "test_end_date", "target_date_end"])
    if start_col is None or end_col is None:
        return {"source": label, "status": "SCHEMA_UNCLEAR", "test_start": None, "test_end": None}
    starts, ends = df[start_col].dropna().astype(str), df[end_col].dropna().astype(str)
    if starts.empty or ends.empty:
        return {"source": label, "status": "DATES_MISSING", "test_start": None, "test_end": None}
    return {"source": label, "status": "AVAILABLE", "test_start": starts.min(), "test_end": ends.max()}

split_df = pd.DataFrame([
    split_range(frames["day10_splits"], "Day 10 classical ML"),
    split_range(frames["day12_splits"], "Day 12 deep learning")
])
display(split_df)

if (split_df["status"] == "AVAILABLE").all():
    a, b = split_df.iloc[0], split_df.iloc[1]
    if a["test_start"] == b["test_start"] and a["test_end"] == b["test_end"]:
        comparability_note = "Reported test date ranges match. Verify per-target sample counts and sample identities before a definitive claim."
    else:
        comparability_note = "Test date ranges differ. Treat scores as indicative, not a controlled head-to-head comparison."
else:
    comparability_note = "Could not verify matching test periods from available reports. Do not declare a definitive winner."

interpretation_rows = []
for target, group in comparison_df.groupby("target", sort=False):
    valid = group.dropna(subset=["MAE", "RMSE"])
    if len(valid) < 2:
        interpretation = "Insufficient comparable metrics"
    elif "differ" in comparability_note.lower() or "could not verify" in comparability_note.lower():
        interpretation = "No definitive winner — test-period comparability unresolved"
    else:
        best = valid.sort_values(["MAE", "RMSE"]).iloc[0]
        interpretation = f"Lower reported MAE: {best['model_family']} ({best['model']})"
    interpretation_rows.append({"target": target, "interpretation": interpretation})
interpretation_df = pd.DataFrame(interpretation_rows)
display(interpretation_df)
print(comparability_note)

,source,status,test_start,test_end
0,Day 10 classical ML,AVAILABLE,2024-04-08,2026-05-29
1,Day 12 deep learning,AVAILABLE,2008-05-03,2026-05-31


,target,interpretation
0,next_match_runs,No definitive winner — test-period comparabili...
1,next_match_wickets,No definitive winner — test-period comparabili...
2,next_match_runs_conceded,No definitive winner — test-period comparabili...


Test date ranges differ. Treat scores as indicative, not a controlled head-to-head comparison.


## 5. Export comparison reports and documentation

The report uses the actual files available at execution time. Missing metrics are left missing, not filled with example or expected values.

In [6]:
comparison_df.to_csv(DAY13_DIR / "ml_vs_dl_model_comparison.csv", index=False)
split_df.to_csv(DAY13_DIR / "test_period_comparability.csv", index=False)
interpretation_df.to_csv(DAY13_DIR / "comparison_interpretation.csv", index=False)
source_status.to_csv(DAY13_DIR / "input_status.csv", index=False)

def frame_text(frame):
    return frame.to_string(index=False) if frame is not None and not frame.empty else "No rows produced."

doc = f"""# Day 13 — ML vs Deep Learning Comparison

## Input availability
```text
{frame_text(source_status)}
```

## Final candidate metrics
```text
{frame_text(comparison_df)}
```

## Test period review
```text
{frame_text(split_df)}
```

## Interpretation
```text
{frame_text(interpretation_df)}
```

**Comparability note:** {comparability_note}

## Outputs
- `reports/day13/ml_vs_dl_model_comparison.csv`
- `reports/day13/test_period_comparability.csv`
- `reports/day13/comparison_interpretation.csv`
- `reports/day13/input_status.csv`
- `docs/day13_ml_vs_dl_comparison.md`

## Method and limitations
- Day 10 is the selected classical ML candidate's final test metrics.
- Day 12 is the selected LSTM/GRU candidate's final test metrics.
- Day 8 provides baseline context; validation and test metrics are not interchangeable.
- Metrics: MAE, RMSE, and R².
- Lower MAE/RMSE and higher R² are generally preferable, but comparison requires compatible test populations.
- A row/sample-level chronological split may place different players from the same match across partition boundaries.
- No performance values or winners are hardcoded.

## Next
Day 14: robustness review, deep-learning error analysis, and candidate-model decision.
"""
doc_path = DOCS_DIR / "day13_ml_vs_dl_comparison.md"
doc_path.write_text(doc, encoding="utf-8")
print("Saved docs:", doc_path)
print("Saved reports in:", DAY13_DIR)

Saved docs: D:\Git\Personal\ipl-player-performance\docs\day13_ml_vs_dl_comparison.md
Saved reports in: D:\Git\Personal\ipl-player-performance\reports\day13
